In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split , GridSearchCV , KFold , cross_val_score
from sklearn.linear_model import LinearRegression 
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor , StackingRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score , mean_squared_error , mean_absolute_error
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler , OneHotEncoder , OrdinalEncoder , PowerTransformer
from sklearn.base import BaseEstimator , TransformerMixin
import optuna
from lightgbm import LGBMRegressor
import mlflow
import dagshub
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate , plot_slice

C:\Users\LeoML\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dagshub.init(repo_owner="mridul0010" , repo_name="Delivery-Time-Analysis-Prediction" , mlflow=True)

Accessing as mridul0010

Initialized MLflow to track repo "mridul0010/Delivery-Time-Analysis-Prediction"

Repository mridul0010/Delivery-Time-Analysis-Prediction initialized!

In [3]:
mlflow.set_tracking_uri("https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow")

In [4]:
mlflow.set_experiment("Stacking Regressor Hyperparameter Tuning")

2026/07/02 20:50:38 INFO mlflow.tracking.fluent: Experiment with name 'Stacking Regressor Hyperparameter Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/056dbc49e4cc47328f10be91bffc7e3a', creation_time=1783005640881, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1783005640881, lifecycle_stage='active', name='Stacking Regressor Hyperparameter Tuning', tags={}, trace_location=None, workspace='default'>

In [ ]:
df = pd.read_csv('../data/Cleaned Delivery Dataset.csv')

In [6]:
pd.set_option('display.max_columns' , None)

In [7]:
df.drop(columns=['Delivery_Agent'] , inplace=True)

In [8]:
df['Order_Datetime'] = pd.to_datetime(df['Order_Datetime'])
df['Pickup_Datetime'] = pd.to_datetime(df['Pickup_Datetime'])

In [9]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        self.rating_bins = None

    def fit(self, X, y=None):
        X = X.copy()
        
        # Store bins for consistent transformation
        self.rating_bins = pd.qcut(
            X['Delivery_person_Ratings'],
            q=3,
            retbins=True,
            duplicates='drop'
        )[1]
        
        return self

    def transform(self, X):
        X = X.copy()

        # Datetime conversion
        X['Order_Datetime'] = pd.to_datetime(X['Order_Datetime'], errors='coerce')
        X['Pickup_Datetime'] = pd.to_datetime(X['Pickup_Datetime'], errors='coerce')

        # Rating group
        X["delivery_rating_group"] = pd.cut(
            X['Delivery_person_Ratings'],
            bins=self.rating_bins,
            labels=['Low', 'Medium', 'High'],
            include_lowest=True
        )

        # Age group
        X["age_group"] = pd.cut(
            X['Delivery_person_Age'],
            bins=[14, 25, 35, 60],
            labels=['Young', 'Adult', 'Senior']
        )

        # Distance group
        X["distance_group"] = pd.cut(
            X['distance_km'],
            bins=[0, 5, 10, 25],
            labels=['Short Distance', 'Medium Distance', 'Long Distance']
        )

        # Time features
        X['Prep_Time(min)'] = (
            X['Pickup_Datetime'] - X['Order_Datetime']
        ).dt.total_seconds() / 60

        X['Order_hour'] = X['Order_Datetime'].dt.hour
        X['Order_day'] = X['Order_Datetime'].dt.day_name()

        X['isWeekend'] = X['Order_day'].isin(["Saturday", "Sunday"]).astype(int)

        X['Time_Of_Day'] = pd.cut(
            X['Order_hour'],
            bins=[0, 6, 12, 18, 24],
            labels=["Night", "Morning", "Afternoon", "Evening"],
            include_lowest=True
        )

        # Drop raw datetime
        X = X.drop(columns=['Order_Datetime', 'Pickup_Datetime'], errors='ignore')

        return X

    def get_feature_names_out(self , input_features = None):
        return input_features

In [10]:
X = df.drop(columns=['Time_taken (min)'])
y = df['Time_taken (min)']

In [11]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [12]:
X

,City,Zone,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,distance_km,Order_Datetime,Pickup_Datetime,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City_Type
0,Dehradun,Zone17,36,4.2,30.327968,78.046106,30.397968,78.116106,10.28,2022-12-02 21:55:00,2022-12-02 22:10:00,Fog,Jam,Good,Snack,motorcycle,3,No,Metropolitian
1,Kochi,Zone16,21,4.7,10.003064,76.307589,10.043064,76.347589,6.24,2022-02-13 14:55:00,2022-02-13 15:05:00,Stormy,High,Average,Meal,motorcycle,1,No,Metropolitian
2,Pune,Zone13,23,4.7,18.562450,73.916619,18.652450,74.006619,13.79,2022-04-03 17:30:00,2022-04-03 17:40:00,Sandstorms,Medium,Average,Drinks,scooter,1,No,Metropolitian
3,Ludhiana,Zone15,34,4.3,30.899584,75.809346,30.919584,75.829346,2.93,2022-02-13 09:20:00,2022-02-13 09:30:00,Sandstorms,Low,poor,Buffet,motorcycle,0,No,Metropolitian
4,Kanpur,Zone14,24,4.7,26.463504,80.372929,26.593504,80.502929,19.40,2022-02-14 19:50:00,2022-02-14 20:05:00,Fog,Jam,Average,Snack,scooter,1,No,Metropolitian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39637,Bangalore,Zone16,28,4.9,13.029198,77.570997,13.059198,77.600997,4.66,2022-03-30 21:55:00,2022-03-30 22:05:00,Sandstorms,Jam,Average,Meal,scooter,1,No,Metropolitian
39638,Ranchi,Zone16,35,4.2,23.371292,85.327872,23.481292,85.437872,16.60,2022-08-03 21:45:00,2022-08-03 21:55:00,Windy,Jam,Good,Drinks,motorcycle,1,No,Metropolitian
39639,Chennai,Zone08,30,4.9,13.022394,80.242439,13.052394,80.272439,4.66,2022-11-03 23:50:00,2022-11-04 00:05:00,Cloudy,Low,Average,Drinks,scooter,0,No,Metropolitian
39640,Coimbatore,Zone11,20,4.7,11.001753,76.986241,11.041753,77.026241,6.23,2022-07-03 13:35:00,2022-07-03 13:40:00,Cloudy,High,poor,Snack,motorcycle,1,No,Metropolitian


In [13]:
numerical = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 
       'multiple_deliveries', 'distance_km', 'Prep_Time(min)', 'Order_hour',
       'isWeekend']

ohe_col = ['City', 'Zone', 'Weather_conditions',
       'Type_of_order', 'Type_of_vehicle',
       'City_Type','Order_day', 'Time_Of_Day']

ordinal_col = ['Road_traffic_density' , 'Vehicle_condition' , 'Festival',
       'delivery_rating_group' , 'age_group' , 'distance_group']

In [14]:
transformer = ColumnTransformer(
    transformers=[
        ("ohe" , OneHotEncoder(sparse_output=False , handle_unknown='ignore') , ohe_col),
        ('oe' , OrdinalEncoder(categories=[
            ['Low', 'Medium', 'High', 'Jam'],                         # Road_traffic_density
            ['poor', 'Average', 'Good' , 'Excellent'],                # Vehicle_condition
            ['No', 'Yes'],                                            # Festival
            ['Low', 'Medium', 'High'],                                # delivery_rating_group
            ['Young', 'Adult', 'Senior'],                             # age_group
            ['Short Distance', 'Medium Distance', 'Long Distance']    # distance_group
        ]) , ordinal_col),
        ('Scaling' , StandardScaler() , numerical)
    ],remainder='passthrough'
)

In [15]:
pipeline = Pipeline(
    [
        ("FeatureEngineering" , FeatureEngineering()),
        ("ColumnTransformer" , transformer)
    ]
) 

In [16]:
X_train_trans = pipeline.fit_transform(X_train)
X_test_trans = pipeline.transform(X_test)

In [17]:
pt = PowerTransformer()

In [18]:
best_xgb_params = {
    'n_estimators': 245,
    'max_depth': 23,
    'learning_rate': 0.19864909172366368,
    'subsample': 0.9180765181845632,
    'min_child_weight': 7,
    'gamma': 1.0407049003047555,
    'reg_lambda': 22.40266224283313
}

best_rf_params = {
    'n_estimators': 299,
    'max_depth': 13,
    'min_samples_split': 20,
    'min_samples_leaf': 2,
    'max_features': 0.44603640869435546,
    'bootstrap': False
}

best_rf = RandomForestRegressor(**best_rf_params)
best_xgb = XGBRegressor(**best_xgb_params)

In [19]:
def objective(trial):
    with mlflow.start_run(nested=True):
        meta_model_name = trial.suggest_categorical("model",["LR","KNN","DT"])

        if meta_model_name == "LR":
            meta = LinearRegression()

        elif meta_model_name == "KNN":
            n_neighbors_knn = trial.suggest_int("n_neighbors_knn",1,15)
            weights_knn = trial.suggest_categorical("weights_knn",["uniform","distance"])
            meta = KNeighborsRegressor(n_neighbors=n_neighbors_knn,
                                        weights=weights_knn,n_jobs=-1)

        elif meta_model_name == "DT":
            max_depth_dt = trial.suggest_int("max_depth_dt",1,10)
            min_samples_split_dt = trial.suggest_int("min_samples_split_dt",2,10)
            min_samples_leaf_dt = trial.suggest_int("min_samples_leaf_dt",1,10)
            meta = DecisionTreeRegressor(max_depth=max_depth_dt,
                                        min_samples_split=min_samples_split_dt,
                                        min_samples_leaf=min_samples_leaf_dt,
                                        random_state=42)

        # log meta model name
        mlflow.log_param("meta_model_name",meta_model_name)

        # stacking regressor
        stacking_reg = StackingRegressor(estimators=[("rf",best_rf),
                                                    ("xgb",best_xgb)],
                                        final_estimator=meta,
                                        cv=5,n_jobs=-1)

        # build transformed regressor
        model = TransformedTargetRegressor(regressor=stacking_reg,
                                            transformer=pt)

        # train the model
        model.fit(X_train_trans,y_train)

        # get the predictions
        y_pred_test = model.predict(X_test_trans)

        # mean absoulte error
        error = mean_absolute_error(y_test,y_pred_test)

        # log error
        mlflow.log_metric("MAE",error)

        return error

In [20]:
# create optuna study
study = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective,n_trials=20,n_jobs=1,show_progress_bar=True)

    # log the best parameters
    mlflow.log_params(study.best_params)

    # log the best score
    mlflow.log_metric("best_score",study.best_value)

[I 2026-07-02 20:50:39,906] A new study created in memory with name: no-name-6c0bfcab-9ae0-4cbd-b061-08efb19334c6
  0%|          | 0/20 [00:00<?, ?it/s]

🏃 View run serious-fawn-757 at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/3/runs/f8d4b9eb867a4982944f06b4349a8e6f
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/3


Best trial: 0. Best value: 3.07617:   5%|▌         | 1/20 [01:17<24:25, 77.15s/it]

[I 2026-07-02 20:51:57,693] Trial 0 finished with value: 3.0761725124314308 and parameters: {'model': 'KNN', 'n_neighbors_knn': 14, 'weights_knn': 'distance'}. Best is trial 0 with value: 3.0761725124314308.
🏃 View run upset-cod-789 at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/3/runs/87cfde6ceb9f4174ac611817e32de081
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/3


Best trial: 0. Best value: 3.07617:  10%|█         | 2/20 [02:33<23:04, 76.94s/it]

...

[I 2026-07-02 21:13:26,595] Trial 17 finished with value: 2.9828093959139417 and parameters: {'model': 'LR'}. Best is trial 2 with value: 2.9819038679610337.
🏃 View run vaunted-goose-261 at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/3/runs/d78c76dda2ca40fe865e1edfb46ac485
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/3


Best trial: 2. Best value: 2.9819:  95%|█████████▌| 19/20 [24:01<01:15, 75.41s/it]

[I 2026-07-02 21:14:42,370] Trial 18 finished with value: 2.982097868689029 and parameters: {'model': 'LR'}. Best is trial 2 with value: 2.9819038679610337.
🏃 View run glamorous-carp-570 at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/3/runs/4e057e2ae7b745a58136b43fa1328109
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/3


Best trial: 2. Best value: 2.9819: 100%|██████████| 20/20 [25:15<00:00, 75.75s/it]


[I 2026-07-02 21:15:55,585] Trial 19 finished with value: 2.9830834140603777 and parameters: {'model': 'LR'}. Best is trial 2 with value: 2.9819038679610337.
🏃 View run best_model at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/3/runs/7e7970d1da54411c80ed43ba25e51632
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/3


In [21]:
study.best_value

2.9819038679610337

In [22]:
study.best_params

{'model': 'LR'}

In [23]:
study.trials_dataframe().head()

,number,value,datetime_start,datetime_complete,duration,params_max_depth_dt,params_min_samples_leaf_dt,params_min_samples_split_dt,params_model,params_n_neighbors_knn,params_weights_knn,state
0,0,3.076173,2026-07-02 20:50:40.543634,2026-07-02 20:51:57.693245,0 days 00:01:17.149611,NaN,NaN,NaN,KNN,14.0,distance,COMPLETE
1,1,3.107926,2026-07-02 20:51:57.698701,2026-07-02 20:53:14.490928,0 days 00:01:16.792227,NaN,NaN,NaN,KNN,10.0,uniform,COMPLETE
2,2,2.981904,2026-07-02 20:53:14.494145,2026-07-02 20:54:51.944199,0 days 00:01:37.450054,NaN,NaN,NaN,LR,NaN,NaN,COMPLETE
3,3,3.241067,2026-07-02 20:54:51.970197,2026-07-02 20:56:11.540002,0 days 00:01:19.569805,NaN,NaN,NaN,KNN,5.0,distance,COMPLETE
4,4,2.982827,2026-07-02 20:56:11.543222,2026-07-02 20:57:25.676527,0 days 00:01:14.133305,NaN,NaN,NaN,LR,NaN,NaN,COMPLETE


In [24]:
study.trials_dataframe()['params_model'].value_counts()

params_model
LR     12
KNN     5
DT      3
Name: count, dtype: int64